# Rice-pest detector — D-FINE-L (COCO weights) fine-tune (Colab T4)
Generated by `tools/build_notebook.py`. Runtime: T4 GPU. Run all; re-running after a disconnect resumes from Drive.

In [ ]:
import os, shutil
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/rice-pest-detector/run-dfine-l'
os.makedirs(RUN_DIR, exist_ok=True)
free_gb = shutil.disk_usage('/content/drive').free / 1e9
print(f'Drive free space: {free_gb:.1f} GB')
if free_gb < 20:
    print('WARNING: less than 20 GB free on Drive; checkpoints may fail to save. Free space first.')

In [ ]:
!nvidia-smi
import torch, sys
print(sys.version, torch.__version__, torch.version.cuda)
!pip -q install transformers==5.20.1 pycocotools scipy

In [ ]:
import transformers
print('transformers', transformers.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('No GPU: choose Runtime > Change runtime type > T4 GPU, then Run all')

In [ ]:
import subprocess, sys, datetime
from pathlib import Path
import torch, transformers
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
def _run(*cmd):
    done = subprocess.run(cmd, capture_output=True, text=True)
    return done.stdout + done.stderr
log = '\n'.join([
    f'utc: {stamp}',
    f'python: {sys.version}',
    f'torch: {torch.__version__} (cuda {torch.version.cuda})',
    f'transformers: {transformers.__version__}',
    '--- nvidia-smi ---', _run('nvidia-smi'),
    '--- pip freeze ---', _run(sys.executable, '-m', 'pip', 'freeze'),
])
Path(RUN_DIR, f'env-{stamp}.txt').write_text(log)
print('environment log written to', f'env-{stamp}.txt')

In [ ]:
import os
os.makedirs('/content/rpd/rice_pest_detector', exist_ok=True)
os.makedirs('/content/rpd/data', exist_ok=True)

In [ ]:
%%writefile /content/rpd/rice_pest_detector/__init__.py
"""Rice black bug / white stemborer detector (RT-DETR) for light-trap images."""

__version__ = "0.1.0"

CLASS_NAMES: tuple[str, ...] = ("RBB_FEMALE", "RBB_MALE", "WSB_FEMALE", "WSB_MALE")
SPECIES_OF_CLASS: dict[int, str] = {0: "RBB", 1: "RBB", 2: "WSB", 3: "WSB"}


In [ ]:
%%writefile /content/rpd/rice_pest_detector/dedup.py
"""Perceptual (dHash) duplicate screening across image families."""

import numpy as np
from PIL import Image


def dhash(image: Image.Image) -> int:
    small = np.asarray(image.convert("L").resize((9, 8), Image.Resampling.BILINEAR), dtype=np.int16)
    bits = (small[:, 1:] > small[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)


def _popcount(x: np.ndarray) -> np.ndarray:
    return np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)


def group_families(hashes: dict[str, list[int]], max_distance: int = 4) -> list[frozenset[str]]:
    families = sorted(hashes)
    owner = np.array([i for i, f in enumerate(families) for _ in hashes[f]], dtype=np.int64)
    values = np.array([h for f in families for h in hashes[f]], dtype=np.uint64)
    parent = list(range(len(families)))

    def find(i: int) -> int:
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for k in range(len(values)):
        close = np.nonzero(_popcount(values ^ values[k]) <= max_distance)[0]
        for j in close:
            ra, rb = find(int(owner[k])), find(int(owner[j]))
            if ra != rb:
                parent[rb] = ra
    groups: dict[int, set[str]] = {}
    for i, f in enumerate(families):
        groups.setdefault(find(i), set()).add(f)
    return [frozenset(g) for g in groups.values()]


In [ ]:
%%writefile /content/rpd/rice_pest_detector/families.py
"""Map Roboflow export filenames to their source-image family."""

import re

_ROBOFLOW = re.compile(
    r"^(?P<stem>.+?)_(?:jpe?g|png)\.rf\.[0-9a-f]+\.(?:jpe?g|png)$",
    re.IGNORECASE,
)


def family_key(filename: str) -> str:
    """Return the lower-cased source stem shared by every export variant of one photo."""
    match = _ROBOFLOW.match(filename)
    if match is None:
        raise ValueError(f"not a Roboflow export filename: {filename!r}")
    return match.group("stem").lower()


In [ ]:
%%writefile /content/rpd/rice_pest_detector/fetch.py
"""Single-request archive download and manifest-scoped extraction (Colab data path)."""

import hashlib
import urllib.request
import zipfile
from pathlib import Path

ZENODO_URL = "https://zenodo.org/api/records/20066074/files/datasets.zip/content"
ZENODO_BYTES = 1_005_857_218
ZENODO_MD5 = "24960c245a3a21b63be95314cc7467da"
ZENODO_SHA256 = "f4a0409977e0615e34f3cf2b936a5ca9dd3a91b153d239e314334ea40c2537f7"


def download(url: str, dest: Path, expected_bytes: int, expected_md5: str, expected_sha256: str) -> Path:
    dest = Path(dest)
    part = dest.with_name(dest.name + ".part")
    md5, sha, size = hashlib.md5(), hashlib.sha256(), 0
    try:
        with urllib.request.urlopen(url, timeout=120) as response, part.open("wb") as out:
            while chunk := response.read(1 << 20):
                out.write(chunk)
                md5.update(chunk)
                sha.update(chunk)
                size += len(chunk)
    except BaseException:
        part.unlink(missing_ok=True)
        raise
    problems = []
    if size != expected_bytes:
        problems.append(f"bytes {size} != {expected_bytes}")
    if md5.hexdigest() != expected_md5:
        problems.append("md5 mismatch")
    if sha.hexdigest() != expected_sha256:
        problems.append("sha256 mismatch")
    if problems:
        part.unlink(missing_ok=True)
        raise ValueError(f"{url}: " + "; ".join(problems))
    part.replace(dest)
    return dest


def extract_members(archive: Path, manifest: dict, out_root: Path) -> int:
    out_root = Path(out_root).resolve()
    count = 0
    with zipfile.ZipFile(archive) as z:
        for rec in manifest["images"]:
            target = (out_root / rec["relative_path"]).resolve()
            if out_root not in target.parents:
                raise ValueError(f"{rec['relative_path']} escapes {out_root}")
            try:
                data = z.read(rec["archive_member"])
            except KeyError:
                raise KeyError(
                    f"archive member {rec['archive_member']!r} not found in {archive}; "
                    "check the archive prefix (see docs/evaluation-record.md, 'Colab data path')"
                ) from None
            if hashlib.sha256(data).hexdigest() != rec["sha256"]:
                raise ValueError(f"digest mismatch for {rec['archive_member']}")
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
            count += 1
    return count


In [ ]:
%%writefile /content/rpd/rice_pest_detector/infer.py
"""Inference contract shared by export parity checks and downstream tools."""

from dataclasses import dataclass

import numpy as np
from PIL import Image, ImageOps

INPUT_SIZE = 640


@dataclass(frozen=True)
class Detection:
    class_id: int
    score: float
    x0: float
    y0: float
    x1: float
    y1: float


def preprocess(image: Image.Image, size: int = INPUT_SIZE) -> np.ndarray:
    """size is the model input side; the app contract is 640 (RF-DETR-M trains at 576)."""
    rgb = ImageOps.exif_transpose(image).convert("RGB")
    resized = rgb.resize((size, size), Image.Resampling.BILINEAR)
    array = np.asarray(resized, dtype=np.float32) / 255.0
    return array.transpose(2, 0, 1)[np.newaxis].copy()


def postprocess(logits: np.ndarray, boxes: np.ndarray, width: int, height: int,
                threshold: float = 0.5) -> list[Detection]:
    """Turn raw RT-DETR outputs into detections in original-image pixels.

    ``width`` and ``height`` MUST be the size of the EXIF-transposed image
    (``ImageOps.exif_transpose(img).size``), not the raw ``img.size``; otherwise
    boxes are wrong on rotated JPEGs.
    """
    probs = 1.0 / (1.0 + np.exp(-logits[0].astype(np.float64)))
    scores, classes = probs.max(axis=1), probs.argmax(axis=1)
    detections = []
    for q in np.nonzero(scores >= threshold)[0]:
        cx, cy, w, h = boxes[0, q].astype(np.float64)
        detections.append(Detection(int(classes[q]), float(scores[q]),
                                    float((cx - w / 2) * width), float((cy - h / 2) * height),
                                    float((cx + w / 2) * width), float((cy + h / 2) * height)))
    return sorted(detections, key=lambda d: -d.score)


In [ ]:
%%writefile /content/rpd/rice_pest_detector/labels.py
"""YOLO-format label parsing and validation."""

import math
from dataclasses import dataclass
from pathlib import Path

from rice_pest_detector import CLASS_NAMES

_EPS = 1e-6


@dataclass(frozen=True)
class Box:
    class_id: int
    cx: float
    cy: float
    w: float
    h: float


def parse_labels(text: str) -> list[Box]:
    boxes: list[Box] = []
    for number, line in enumerate(text.splitlines(), start=1):
        fields = line.split()
        if not fields:
            continue
        if len(fields) == 5:
            boxes.append(Box(int(fields[0]), *(float(v) for v in fields[1:])))
        elif len(fields) >= 7 and len(fields) % 2 == 1:
            # YOLO-seg polygon: class id followed by x,y pairs -> min/max bounding box.
            values = [float(v) for v in fields[1:]]
            xs, ys = values[0::2], values[1::2]
            boxes.append(Box(int(fields[0]), (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2,
                             max(xs) - min(xs), max(ys) - min(ys)))
        else:
            raise ValueError(f"line {number}: expected 5 fields or an odd polygon row, got {len(fields)}")
    return boxes


def validate_boxes(boxes: list[Box]) -> list[str]:
    errors: list[str] = []
    for i, b in enumerate(boxes):
        if not 0 <= b.class_id < len(CLASS_NAMES):
            errors.append(f"box {i}: class {b.class_id} not in 0..{len(CLASS_NAMES) - 1}")
        elif not all(math.isfinite(v) for v in (b.cx, b.cy, b.w, b.h)):
            errors.append(f"box {i}: non-finite coordinate")
        elif b.w <= 0 or b.h <= 0:
            errors.append(f"box {i}: non-positive area")
        elif (b.cx - b.w / 2 < -_EPS or b.cx + b.w / 2 > 1 + _EPS
              or b.cy - b.h / 2 < -_EPS or b.cy + b.h / 2 > 1 + _EPS):
            errors.append(f"box {i}: extends outside the image")
    return errors


def load_label_file(path: Path) -> list[Box]:
    return parse_labels(Path(path).read_text(encoding="utf-8"))


In [ ]:
%%writefile /content/rpd/rice_pest_detector/manifest.py
"""Build the leakage-free split manifest and COCO annotations from the local dataset."""

import argparse
import hashlib
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
from PIL import Image

from rice_pest_detector import CLASS_NAMES
from rice_pest_detector.dedup import dhash, group_families
from rice_pest_detector.families import family_key
from rice_pest_detector.labels import Box, load_label_file, validate_boxes
from rice_pest_detector.split import Variant, select_eval_variant, split_groups, stratum

_SPLITS = ("train", "valid", "test")

# Measured on the real dataset: see "Data preparation" in docs/evaluation-record.md.
DEDUP_MAX_DISTANCE = 1


def _representative(items: list[dict]) -> tuple[dict, str | None]:
    """The family's representative image, chosen by the evaluation-variant rule."""
    variants = [Variant(i["path"].name, i["origin"], i["grey"]) for i in items]
    pick, rule = select_eval_variant(variants)
    return next(i for i in items if i["path"].name == pick.name), rule


def _is_grey(img: Image.Image) -> bool:
    a = np.asarray(img.convert("RGB"), dtype=np.int16)
    return bool(np.abs(a[..., 0] - a[..., 1]).max() <= 2 and np.abs(a[..., 1] - a[..., 2]).max() <= 2)


def _scan(root: Path) -> tuple[dict[str, list[dict]], list[dict]]:
    families: dict[str, list[dict]] = defaultdict(list)
    for origin in _SPLITS:
        image_dir = root / "02_detection_dataset" / origin / "images"
        for path in sorted(image_dir.glob("*")):
            if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                continue
            families[family_key(path.name)].append({"path": path, "origin": origin})
    exclusions: list[dict] = []
    for fam in sorted(families):
        for item in families[fam]:
            label = item["path"].parent.parent / "labels" / (item["path"].stem + ".txt")
            try:
                item["boxes"] = load_label_file(label)
            except FileNotFoundError:
                exclusions.append({"family": fam, "reason": f"missing label for {item['path'].name}"})
                break
            except ValueError as exc:
                exclusions.append({"family": fam, "reason": f"{item['path'].name}: {exc}"})
                break
            errors = validate_boxes(item["boxes"])
            if errors:
                exclusions.append({"family": fam, "reason": f"{item['path'].name}: {errors[0]}"})
                break
    for e in exclusions:
        families.pop(e["family"], None)
    return families, exclusions


def _coco(records: list[dict]) -> dict:
    images, annotations = [], []
    for image_id, rec in enumerate(records, start=1):
        w, h = rec["width"], rec["height"]
        images.append({"id": image_id, "file_name": rec["file"], "width": w, "height": h})
        for c, cx, cy, bw, bh in rec["boxes"]:
            box = [(cx - bw / 2) * w, (cy - bh / 2) * h, bw * w, bh * h]
            annotations.append({"id": len(annotations) + 1, "image_id": image_id, "category_id": c + 1,
                                "bbox": box, "area": box[2] * box[3], "iscrowd": 0})
    categories = [{"id": i + 1, "name": n} for i, n in enumerate(CLASS_NAMES)]
    return {"images": images, "annotations": annotations, "categories": categories}


def prepare(dataset_root: Path, out_dir: Path, seed: int = 20261013, num_queries: int = 300) -> dict:
    root = Path(dataset_root)
    families, exclusions = _scan(root)
    for fam, items in families.items():
        for item in items:
            if len(item["boxes"]) > num_queries:
                raise ValueError(f"{item['path'].name}: {len(item['boxes'])} boxes exceeds "
                                 f"num_queries={num_queries}")
            with Image.open(item["path"]) as img:
                item["grey"], item["size"] = _is_grey(img), img.size
                item["hash"] = dhash(img)
    rep_hashes = {f: [_representative(items)[0]["hash"]] for f, items in families.items()}
    groups = group_families(rep_hashes, max_distance=DEDUP_MAX_DISTANCE)
    group_of = {f: f"grp{n:05d}" for n, g in enumerate(sorted(groups, key=sorted)) for f in g}
    group_boxes: dict[str, list[Box]] = defaultdict(list)
    for fam, items in families.items():
        group_boxes[group_of[fam]].extend(b for i in items for b in i["boxes"])
    split_of_group = split_groups({g: stratum(b) for g, b in group_boxes.items()}, seed=seed)

    records: list[dict] = []
    for fam in sorted(families):
        items, split = families[fam], split_of_group[group_of[fam]]
        chosen, rule = items, None
        if split != "train":
            pick, rule = _representative(items)
            chosen = [pick]
        for item in chosen:
            rel = item["path"].relative_to(root).as_posix()
            data = item["path"].read_bytes()
            records.append({
                "file": item["path"].name, "relative_path": rel, "archive_member": f"datasets/{rel}",
                "family": fam, "group": group_of[fam], "split": split, "origin_split": item["origin"],
                "selection_rule": rule, "sha256": hashlib.sha256(data).hexdigest(), "bytes": len(data),
                "width": item["size"][0], "height": item["size"][1],
                "boxes": [[b.class_id, b.cx, b.cy, b.w, b.h] for b in item["boxes"]],
            })

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    (out / "split_manifest.json").write_text(json.dumps({"seed": seed, "images": records}, indent=1),
                                             encoding="utf-8", newline="\n")
    (out / "exclusions.json").write_text(json.dumps(exclusions, indent=1), encoding="utf-8", newline="\n")
    for split in ("train", "val", "test"):
        coco = _coco([r for r in records if r["split"] == split])
        (out / f"annotations_{split}.json").write_text(json.dumps(coco), encoding="utf-8", newline="\n")
    counts = {s: sum(r["split"] == s for r in records) for s in ("train", "val", "test")}
    return {"images": counts, "families": len(families), "groups": len(groups),
            "excluded_families": len(exclusions)}


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--dataset-root", type=Path, required=True)
    parser.add_argument("--out", type=Path, default=Path("data"))
    args = parser.parse_args()
    print(json.dumps(prepare(args.dataset_root, args.out), indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/rpd/rice_pest_detector/metrics.py
"""Ship-gate metrics: species-grouped mAP50 and per-image count accuracy."""

import contextlib
import copy
import io

import numpy as np

from rice_pest_detector import SPECIES_OF_CLASS

_SPECIES = ("RBB", "WSB")


def count_within_tolerance(pred: int, ref: int) -> bool:
    if ref < 10:
        return abs(pred - ref) <= 1
    return abs(pred - ref) <= 0.10 * ref


def species_counts(class_ids: list[int]) -> dict[str, int]:
    counts = dict.fromkeys(_SPECIES, 0)
    for c in class_ids:
        counts[SPECIES_OF_CLASS[c]] += 1
    return counts


def count_accuracy(pred_counts: list[dict[str, int]], ref_counts: list[dict[str, int]]) -> float:
    passed = [all(count_within_tolerance(p[s], r[s]) for s in _SPECIES)
              for p, r in zip(pred_counts, ref_counts, strict=True)]
    return float(np.mean(passed))


_DENSITY_BINS = ((1, 9, "1-9"), (10, 29, "10-29"), (30, None, "30+"))


def count_errors(pred_counts: list[dict[str, int]], ref_counts: list[dict[str, int]]) -> dict:
    """Per-species counting errors to read beside count_accuracy, whose +-1 tolerance below a
    reference of 10 lets an image with one insect pass when nothing is detected.

    mae: mean |pred - ref| over all images; bias: mean (pred - ref), negative = undercount;
    exact: share of images counted exactly; presence_recall: share of images containing the
    species where at least one was counted; mae_by_density: mae over images whose reference
    count falls in each bin (images without the species are excluded).
    """
    out = {}
    for s in _SPECIES:
        pairs = [(p[s], r[s]) for p, r in zip(pred_counts, ref_counts, strict=True)]
        diffs = np.array([p - r for p, r in pairs], dtype=float)
        present = [(p, r) for p, r in pairs if r > 0]
        by_density = {}
        for lo, hi, label in _DENSITY_BINS:
            sub = [abs(p - r) for p, r in present if r >= lo and (hi is None or r <= hi)]
            by_density[label] = {"images": len(sub), "mae": float(np.mean(sub)) if sub else None}
        out[s] = {"mae": float(np.abs(diffs).mean()) if pairs else None,
                  "bias": float(diffs.mean()) if pairs else None,
                  "exact": float(np.mean(diffs == 0)) if pairs else None,
                  "presence_recall": float(np.mean([p > 0 for p, _ in present])) if present else None,
                  "mae_by_density": by_density}
    return out


def _box_iou(a: list[float], b: list[float]) -> float:
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def confusion(gt: list[tuple[int, list[float]]], det: list[tuple[int, list[float]]],
              iou: float = 0.5) -> dict[tuple[str, str], int]:
    from rice_pest_detector import CLASS_NAMES

    table: dict[tuple[str, str], int] = {}
    unmatched = list(range(len(gt)))
    for d_cls, d_box in det:
        best = max(unmatched, key=lambda g: _box_iou(gt[g][1], d_box), default=None)
        if best is not None and _box_iou(gt[best][1], d_box) >= iou:
            unmatched.remove(best)
            key = (CLASS_NAMES[gt[best][0]], CLASS_NAMES[d_cls])
        else:
            key = ("spurious", CLASS_NAMES[d_cls])
        table[key] = table.get(key, 0) + 1
    for g in unmatched:
        key = (CLASS_NAMES[gt[g][0]], "missed")
        table[key] = table.get(key, 0) + 1
    return table


def _to_species(coco: dict, dets: list[dict]) -> tuple[dict, list[dict]]:
    species_id = {c: 1 if SPECIES_OF_CLASS[c - 1] == "RBB" else 2 for c in range(1, 5)}
    gt = copy.deepcopy(coco)
    for a in gt["annotations"]:
        a["category_id"] = species_id[a["category_id"]]
    gt["categories"] = [{"id": 1, "name": "RBB"}, {"id": 2, "name": "WSB"}]
    return gt, [{**d, "category_id": species_id[d["category_id"]]} for d in dets]


def map50(gt_coco: dict, detections: list[dict], group_species: bool) -> dict[str, float]:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    gt_dict, dets = _to_species(gt_coco, detections) if group_species else (gt_coco, detections)
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO()
        gt.dataset = copy.deepcopy(gt_dict)
        gt.createIndex()
        dt = gt.loadRes(dets) if dets else COCO()
        ev = COCOeval(gt, dt, "bbox")
        ev.params.maxDets = [1, 10, 300]  # trays hold up to 106 insects; COCO's default cap is 100
        ev.evaluate()
        ev.accumulate()
        ev.summarize()
    result = {"all": float(ev.stats[1])}  # AP at IoU 0.50, maxDets=300
    precision = ev.eval["precision"][0, :, :, 0, -1]  # IoU=0.50, all recall, area=all, maxDets=300
    for k, cat in enumerate(gt_dict["categories"]):
        p = precision[:, k]
        result[cat["name"]] = float(p[p > -1].mean()) if (p > -1).any() else 0.0
    return result


In [ ]:
%%writefile /content/rpd/rice_pest_detector/split.py
"""Family-group split (70/15/15, stratified) and evaluation-variant selection."""

import random
from dataclasses import dataclass

from rice_pest_detector import SPECIES_OF_CLASS
from rice_pest_detector.labels import Box


def stratum(boxes: list[Box]) -> str:
    species = {SPECIES_OF_CLASS[b.class_id] for b in boxes}
    if species == {"RBB"}:
        return "rbb"
    if species == {"WSB"}:
        return "wsb"
    return "mixed"


def split_groups(
    group_strata: dict[str, str],
    seed: int = 20261013,
    ratios: tuple[float, float, float] = (0.70, 0.15, 0.15),
) -> dict[str, str]:
    rng = random.Random(seed)
    assignment: dict[str, str] = {}
    for name in sorted(set(group_strata.values())):
        groups = sorted(g for g, s in group_strata.items() if s == name)
        rng.shuffle(groups)
        n_val = round(ratios[1] * len(groups))
        n_test = round(ratios[2] * len(groups))
        for i, g in enumerate(groups):
            assignment[g] = "val" if i < n_val else "test" if i < n_val + n_test else "train"
    return assignment


@dataclass(frozen=True)
class Variant:
    name: str
    origin_split: str
    grayscale: bool


def select_eval_variant(variants: list[Variant]) -> tuple[Variant, str]:
    ordered = sorted(variants, key=lambda v: v.name)
    for v in ordered:
        if v.origin_split in {"valid", "test"}:
            return v, "published-original"
    for v in ordered:
        if not v.grayscale:
            return v, "first-colour"
    return ordered[0], "first-any"


In [ ]:
%%writefile /content/rpd/rice_pest_detector/train.py
"""Fine-tune RT-DETR on the split manifest; resumable checkpoints; validation-only selection."""

import hashlib
import json
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForObjectDetection, RTDetrConfig, RTDetrForObjectDetection

from rice_pest_detector import CLASS_NAMES
from rice_pest_detector.infer import postprocess, preprocess
from rice_pest_detector.metrics import confusion, count_accuracy, count_errors, map50, species_counts

EVALUATOR_VERSION = 2  # bump whenever evaluate() changes what it measures


@dataclass
class TrainConfig:
    base_model: str
    revision: str
    epochs: int = 60
    batch_size: int = 8
    lr: float = 1e-4
    backbone_lr: float = 1e-5
    weight_decay: float = 1e-4
    grad_clip: float = 0.1
    seed: int = 20261013
    num_queries: int = 300
    fp16: bool = True
    num_workers: int = 2
    input_size: int = 640  # model input side; 640 is the app contract


class ManifestDataset(Dataset):
    def __init__(self, manifest: dict, image_root: Path, split: str, size: int = 640):
        self.records = [r for r in manifest["images"] if r["split"] == split]
        self.root = Path(image_root)
        self.size = size

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, i: int):
        rec = self.records[i]
        with Image.open(self.root / rec["relative_path"]) as img:
            pixels = torch.from_numpy(preprocess(img, self.size)[0])
        boxes = rec["boxes"]
        target = {"class_labels": torch.tensor([b[0] for b in boxes], dtype=torch.long),
                  "boxes": torch.tensor([b[1:] for b in boxes], dtype=torch.float32).reshape(-1, 4)}
        return pixels, target


def collate(batch):
    return torch.stack([b[0] for b in batch]), [b[1] for b in batch]


def _tiny_config() -> RTDetrConfig:  # replaced in tests
    raise RuntimeError("tiny config only available in tests")


def build_model(cfg: TrainConfig):
    """RT-DETR or D-FINE: the checkpoint's own config picks the class (same sigmoid logits / cxcywh boxes)."""
    if cfg.base_model == "__tiny__":
        model = RTDetrForObjectDetection(_tiny_config())
    else:
        id2label = dict(enumerate(CLASS_NAMES))
        model = AutoModelForObjectDetection.from_pretrained(
            cfg.base_model, revision=cfg.revision, num_labels=len(CLASS_NAMES), id2label=id2label,
            label2id={v: k for k, v in id2label.items()}, num_queries=cfg.num_queries,
            ignore_mismatched_sizes=True,
        )
    model.loss_function = fp32_loss(model.loss_function)
    return model


def _to_fp32(value):
    if torch.is_tensor(value):
        return value.float() if value.is_floating_point() else value
    if isinstance(value, dict):
        return {k: _to_fp32(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return type(value)(_to_fp32(v) for v in value)
    return value


def fp32_loss(loss_fn):
    """Run RT-DETR's matcher and losses in float32 even under fp16 autocast.

    The Hungarian matcher takes log(p + 1e-8) of sigmoid scores; in float16 the epsilon
    underflows and saturated logits give inf costs, which scipy rejects with ValueError
    before the training loop's finite-loss check can run. The final, auxiliary and encoder
    outputs all pass through this one call.
    """
    def wrapped(*args, **kwargs):
        device_type = "cuda" if torch.cuda.is_available() else "cpu"
        with torch.autocast(device_type=device_type, enabled=False):
            return loss_fn(*_to_fp32(args), **_to_fp32(kwargs))
    wrapped.__wrapped__ = loss_fn
    return wrapped


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _sha256_json(obj) -> str:
    return _sha256_bytes(json.dumps(obj, sort_keys=True).encode("utf-8"))


def experiment_id(cfg: TrainConfig, manifest: dict, val_annotations: bytes) -> dict:
    """What makes two runs the same experiment; num_workers only changes speed."""
    settings = {k: v for k, v in asdict(cfg).items() if k != "num_workers"}
    if settings.get("input_size") == 640:  # runs from before the field existed keep their identity
        del settings["input_size"]
    return {"config": settings, "manifest_sha256": _sha256_json(manifest),
            "annotations_val_sha256": _sha256_bytes(val_annotations)}


def check_experiment(saved: dict | None, current: dict) -> None:
    if saved is None:
        print("WARNING: last.pt predates experiment fingerprints; resuming without an identity check")
        return
    if saved != current:
        changed = sorted(k for k in saved["config"].keys() | current["config"].keys()
                         if saved["config"].get(k) != current["config"].get(k))
        changed += [k for k in ("manifest_sha256", "annotations_val_sha256") if saved.get(k) != current.get(k)]
        raise RuntimeError(f"run directory holds a different experiment (changed: {', '.join(changed)}); "
                           "use a new RUN_DIR instead of resuming")


def _rng_state() -> dict:
    name, keys, pos, has_gauss, cached = np.random.get_state()
    return {"python": random.getstate(), "numpy": [name, keys.tolist(), pos, has_gauss, cached],
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}


def _set_rng_state(state: dict) -> None:
    version, internal, gauss = state["python"]
    random.setstate((version, tuple(internal), gauss))
    name, keys, pos, has_gauss, cached = state["numpy"]
    np.random.set_state((name, np.array(keys, dtype=np.uint32), pos, has_gauss, cached))
    torch.set_rng_state(state["torch"])
    if state["cuda"] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])


def _atomic_save(obj, path: Path) -> None:
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(path)


def _atomic_write_text(path: Path, text: str) -> None:
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text)
    tmp.replace(path)


def save_checkpoint(path: Path, model, optimizer, scheduler, epoch: int, best_metric: float,
                    scaler=None, experiment: dict | None = None) -> None:
    _atomic_save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                  "scheduler": scheduler.state_dict(), "epoch": epoch, "best_metric": best_metric,
                  "scaler": scaler.state_dict() if scaler is not None else {},
                  "experiment": experiment, "rng": _rng_state()}, path)


def load_checkpoint(path: Path, model, optimizer, scheduler, scaler=None,
                    experiment: dict | None = None) -> tuple[int, float]:
    state = torch.load(path, map_location="cpu", weights_only=True)
    if experiment is not None:
        check_experiment(state.get("experiment"), experiment)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    if scaler is not None and state.get("scaler"):
        scaler.load_state_dict(state["scaler"])
    if state.get("rng"):
        _set_rng_state(state["rng"])
    return state["epoch"] + 1, state["best_metric"]


def load_best(path: Path) -> tuple[dict, dict]:
    """Return (state_dict, metadata) from best.pt; older runs saved a bare state_dict."""
    state = torch.load(path, map_location="cpu", weights_only=True)
    if "model" in state and "map50_species" in state:
        return state["model"], {k: v for k, v in state.items() if k != "model"}
    return state, {}


def record_best(run_dir: Path, model, score: float, epoch: int, best: float,
                experiment: dict | None = None) -> float:
    """Save weights, score and epoch together in one atomic best.pt if `score` beats `best`.

    best.json is a human-readable copy written afterwards; resume trusts best.pt, so an
    interruption between the two writes cannot pair new weights with an old score.
    """
    if not math.isfinite(score) or score <= best:
        return best
    run_dir = Path(run_dir)
    _atomic_save({"model": model.state_dict(), "map50_species": score, "epoch": epoch,
                  "experiment": experiment}, run_dir / "best.pt")
    _atomic_write_text(run_dir / "best.json", json.dumps({"map50_species": score, "epoch": epoch}))
    return score


MAX_NONFINITE_STEPS = 50


def read_history(path: Path) -> list[dict]:
    """Parse history.jsonl, dropping a final line cut short by a disconnect; any other bad line raises."""
    lines = [ln for ln in Path(path).read_text().splitlines() if ln.strip()]
    rows = []
    for i, ln in enumerate(lines):
        try:
            rows.append(json.loads(ln))
        except json.JSONDecodeError:
            if i != len(lines) - 1:
                raise
            print(f"WARNING: dropping truncated final history line: {ln[:60]!r}")
    return rows


def trim_history(path: Path, start: int) -> None:
    """Drop history rows with epoch >= start so a resumed run does not duplicate epochs."""
    path = Path(path)
    if not path.exists():
        return
    rows = [r for r in read_history(path) if r["epoch"] < start]
    _atomic_write_text(path, "".join(json.dumps(r) + "\n" for r in rows))


def resume_best(run_dir: Path, ckpt_best: float) -> float:
    """Best metric on resume: last.pt may lag best.pt by one epoch after an interruption."""
    run_dir = Path(run_dir)
    if (run_dir / "best.pt").exists():
        _, meta = load_best(run_dir / "best.pt")
        if meta:
            return max(ckpt_best, meta["map50_species"])
    sidecar = run_dir / "best.json"  # runs from before best.pt carried its own score
    if sidecar.exists():
        return max(ckpt_best, json.loads(sidecar.read_text())["map50_species"])
    return ckpt_best


def _quantiles(values: list[float]) -> dict[str, float]:
    if not values:
        return {}
    q = np.quantile(values, [0.5, 0.9, 1.0])
    return {"p50": float(q[0]), "p90": float(q[1]), "max": float(q[2])}


@torch.no_grad()
def evaluate(model, dataset: ManifestDataset, coco_gt: dict, threshold: float = 0.5) -> dict:
    model.eval()
    device = next(model.parameters()).device
    dets, preds, refs, table, top_scores, kept_total = [], [], [], {}, [], 0
    for image_id, rec in enumerate(dataset.records, start=1):
        assert coco_gt["images"][image_id - 1]["file_name"] == rec["file"], (
            f"COCO image {image_id} is {coco_gt['images'][image_id - 1]['file_name']!r} "
            f"but manifest record is {rec['file']!r}")
        pixels, _ = dataset[image_id - 1]
        out = model(pixel_values=pixels[None].to(device))
        found = postprocess(out.logits.float().cpu().numpy(), out.pred_boxes.float().cpu().numpy(),
                            rec["width"], rec["height"], threshold=0.0)
        dets += [{"image_id": image_id, "category_id": d.class_id + 1, "score": d.score,
                  "bbox": [d.x0, d.y0, d.x1 - d.x0, d.y1 - d.y0]} for d in found]
        top_scores.append(found[0].score if found else 0.0)
        kept = [d for d in found if d.score >= threshold]
        kept_total += len(kept)
        preds.append(species_counts([d.class_id for d in kept]))
        refs.append(species_counts([b[0] for b in rec["boxes"]]))
        w, h = rec["width"], rec["height"]
        gt = [(c, [(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
              for c, cx, cy, bw, bh in rec["boxes"]]
        for key, n in confusion(gt, [(d.class_id, [d.x0, d.y0, d.x1, d.y1]) for d in kept]).items():
            table[key] = table.get(key, 0) + n
    zero = [species_counts([]) for _ in refs]
    return {"map50_species": map50(coco_gt, dets, group_species=True)["all"],
            "map50_by_class": map50(coco_gt, dets, group_species=False),
            "threshold": threshold,
            "count_accuracy": count_accuracy(preds, refs),
            "count_accuracy_no_detection_baseline": count_accuracy(zero, refs),
            "count_errors": count_errors(preds, refs),
            "detections_kept": kept_total,
            "top_score_per_image": _quantiles(top_scores),
            "confusion": {f"{g}->{p}": n for (g, p), n in sorted(table.items())}}


def evaluation_id(best_path: Path, manifest: dict, coco_path: Path, threshold: float) -> dict:
    """Everything a cached test result depends on, not just the weights."""
    return {"best_pt_sha256": _sha256_bytes(Path(best_path).read_bytes()),
            "manifest_sha256": _sha256_json(manifest),
            "annotations_sha256": _sha256_bytes(Path(coco_path).read_bytes()),
            "threshold": threshold, "evaluator_version": EVALUATOR_VERSION}


def check_cached_test(prior: dict, current: dict) -> None:
    """Refuse a cached test result that no longer describes the current evaluation."""
    saved = prior.get("evaluation")
    if saved is None:  # legacy record: bound to the weights only
        if prior.get("best_pt_sha256") != current["best_pt_sha256"]:
            raise RuntimeError("best.pt changed after the test split was evaluated: protocol violation, "
                               "stop and ask the owner")
        return
    changed = [k for k in current if saved.get(k) != current[k]]
    if changed:
        raise RuntimeError(f"test split already evaluated under a different setup (changed: "
                           f"{', '.join(changed)}): protocol violation, stop and ask the owner")


def fit(cfg: TrainConfig, manifest: dict, image_root: Path, run_dir: Path) -> Path:
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)
    torch.manual_seed(cfg.seed)
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = cfg.fp16 and device == "cuda"
    val_bytes = (run_dir / "annotations_val.json").read_bytes()
    experiment = experiment_id(cfg, manifest, val_bytes)
    model = build_model(cfg).to(device)
    backbone = [p for n, p in model.named_parameters() if "backbone" in n]
    rest = [p for n, p in model.named_parameters() if "backbone" not in n]
    opt = torch.optim.AdamW([{"params": rest, "lr": cfg.lr}, {"params": backbone, "lr": cfg.backbone_lr}],
                            weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    scaler = torch.amp.GradScaler(device, enabled=amp)
    start, best = 0, -1.0
    if (run_dir / "last.pt").exists():
        start, best = load_checkpoint(run_dir / "last.pt", model, opt, sched, scaler=scaler,
                                      experiment=experiment)
        best = resume_best(run_dir, best)
        trim_history(run_dir / "history.jsonl", start)
        print(f"resumed at epoch {start}, best val mAP50 {best:.4f}")
    train_set = ManifestDataset(manifest, image_root, "train", size=cfg.input_size)
    if len(train_set) == 0:
        raise ValueError("train split is empty")
    val_set = ManifestDataset(manifest, image_root, "val", size=cfg.input_size)
    val_coco = json.loads(val_bytes)
    _atomic_write_text(run_dir / "config.json", json.dumps({**asdict(cfg), "experiment": experiment}, indent=1))
    for epoch in range(start, cfg.epochs):
        train = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate,
                           num_workers=cfg.num_workers, generator=torch.Generator().manual_seed(cfg.seed + epoch))
        model.train()
        total, steps, updates, bad_loss, bad_grad, bad = 0.0, 0, 0, 0, 0, 0
        step_no = -1
        for pixels, targets in train:
            step_no += 1
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, enabled=amp):
                loss = model(pixel_values=pixels.to(device), labels=targets).loss
            if not torch.isfinite(loss.detach()):
                bad_loss += 1
                bad += 1
            else:
                total += loss.detach().item()
                steps += 1
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                if torch.isfinite(norm):
                    scaler.step(opt)
                    updates += 1
                    bad = 0
                else:  # skip the update; under AMP the scaler also lowers its scale
                    bad_grad += 1
                    bad += 1
                scaler.update()
            if bad >= MAX_NONFINITE_STEPS:
                raise RuntimeError(
                    f"loss or gradient non-finite for {bad} consecutive steps (epoch {epoch}, step {step_no}); "
                    "training diverged, stopping")
        if updates == 0:
            raise RuntimeError(f"epoch {epoch}: no optimizer updates (all losses or gradients non-finite)")
        sched.step()
        metrics = evaluate(model, val_set, val_coco)
        row = {"epoch": epoch, "train_loss": total / steps, "updates": updates,
               "nonfinite_loss_steps": bad_loss, "skipped_updates": bad_grad,
               "grad_scale": scaler.get_scale() if amp else None, **metrics}
        print(json.dumps({k: v for k, v in row.items() if k != "confusion"}))
        with (run_dir / "history.jsonl").open("a") as fh:
            fh.write(json.dumps(row) + "\n")
        best = record_best(run_dir, model, metrics["map50_species"], epoch, best, experiment=experiment)
        save_checkpoint(run_dir / "last.pt", model, opt, sched, epoch, best, scaler=scaler,
                        experiment=experiment)
    return run_dir / "best.pt"


In [ ]:
import base64, gzip, pathlib
B64 = """
...
BwcA
"""
pathlib.Path('/content/rpd/data/annotations_test.json').write_bytes(gzip.decompress(base64.b64decode(''.join(B64.split()))))

In [ ]:
import base64, gzip, pathlib
B64 = """
...
y/Qv9H/8X/7P/wX+vkhguiWJAA==
"""
pathlib.Path('/content/rpd/data/annotations_train.json').write_bytes(gzip.decompress(base64.b64decode(''.join(B64.split()))))

In [ ]:
import base64, gzip, pathlib
B64 = """
...
"""
pathlib.Path('/content/rpd/data/annotations_val.json').write_bytes(gzip.decompress(base64.b64decode(''.join(B64.split()))))

In [ ]:
import base64, gzip, pathlib
B64 = """
H4sIAAAAAAAC/4vmUqjmUlBQSkvMzcypVLJSUMrMTY83MjAwUtIBiRelJhbn54HEczOLizPz0hVyEpNScxTS8osUPH3dwSrjvQLc
9YrS9FKSTJONTI0MzdOMLEwMTE0tjEyBzGRTC8u0RHPLNCO9rIJ0JS6FWq5YABRhWOl2AAAA
"""
pathlib.Path('/content/rpd/data/exclusions.json').write_bytes(gzip.decompress(base64.b64decode(''.join(B64.split()))))

In [ ]:
import base64, gzip, pathlib
B64 = """
...
"""
pathlib.Path('/content/rpd/data/split_manifest.json').write_bytes(gzip.decompress(base64.b64decode(''.join(B64.split()))))

In [ ]:
import sys, json, shutil, hashlib
sys.path.insert(0, '/content/rpd')
from pathlib import Path
from rice_pest_detector import fetch
dest = Path('/content/datasets.zip')
def _pins_ok(p):
    if not p.exists() or p.stat().st_size != fetch.ZENODO_BYTES:
        return False
    md5, sha = hashlib.md5(), hashlib.sha256()
    with p.open('rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            md5.update(chunk)
            sha.update(chunk)
    return md5.hexdigest() == fetch.ZENODO_MD5 and sha.hexdigest() == fetch.ZENODO_SHA256
drive_copy = Path('/content/drive/MyDrive/rice-pest-detector/datasets.zip')
if _pins_ok(dest):
    archive = dest
    print('verified archive present, skipping download')
elif drive_copy.exists():  # fallback copy, verified with the same pins
    archive = fetch.download(drive_copy.as_uri(), dest, fetch.ZENODO_BYTES, fetch.ZENODO_MD5, fetch.ZENODO_SHA256)
else:
    archive = fetch.download(fetch.ZENODO_URL, dest, fetch.ZENODO_BYTES, fetch.ZENODO_MD5, fetch.ZENODO_SHA256)
manifest = json.loads(Path('/content/rpd/data/split_manifest.json').read_text())
print(fetch.extract_members(archive, manifest, Path('/content/data')), 'images verified')
shutil.copy('/content/rpd/data/annotations_val.json', RUN_DIR)

In [ ]:
from rice_pest_detector.train import TrainConfig, fit
cfg = TrainConfig(base_model="ustc-community/dfine-large-coco", revision="aa8f8211545e4cff5adf52ea512f9ac74a9824ec", batch_size=4, epochs=40)
best = fit(cfg, manifest, Path('/content/data'), Path(RUN_DIR))
print(best)

In [ ]:
from rice_pest_detector.train import (ManifestDataset, build_model, check_cached_test, evaluate,
                                      evaluation_id, load_best, _atomic_write_text)
THRESHOLD = None  # set to the threshold chosen on VALIDATION (see docs/evaluation-record.md), then run
if THRESHOLD is None:
    raise RuntimeError('Test look not unlocked: choose the threshold on validation first, then set THRESHOLD.')
test_coco_path = Path('/content/rpd/data/annotations_test.json')
identity = evaluation_id(best, manifest, test_coco_path, THRESHOLD)
metrics_path = Path(RUN_DIR, 'test_metrics.json')
if metrics_path.exists():
    prior = json.loads(metrics_path.read_text())
    check_cached_test(prior, identity)
    print(json.dumps(prior, indent=1))
    print('test split already evaluated for this checkpoint and setup')
else:
    state, meta = load_best(best)
    model = build_model(cfg)
    model.load_state_dict(state)
    model = model.cuda()
    test_set = ManifestDataset(manifest, Path("/content/data"), split="test")
    metrics = evaluate(model, test_set, json.loads(test_coco_path.read_text()), threshold=THRESHOLD)
    metrics = {**metrics, 'best_pt_sha256': identity['best_pt_sha256'], 'epoch': meta['epoch'],
               'evaluation': identity}
    print(json.dumps(metrics, indent=1))
    _atomic_write_text(metrics_path, json.dumps(metrics, indent=1))